In [ ]:
#import python packages
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from datetime import datetime
import glob
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.optimize import least_squares, differential_evolution
from scipy.integrate import solve_ivp
import time as timer
import random

# ============================================================
# SET RANDOM SEEDS FOR REPRODUCIBILITY
# ============================================================
RANDOM_SEED = 42

# Python random
random.seed(RANDOM_SEED)

# NumPy
np.random.seed(RANDOM_SEED)

# PyTorch
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)  # for multi-GPU

# PyTorch backends
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("="*70)
print("REPRODUCIBILITY SETTINGS")
print("="*70)
print(f"Random seed set to: {RANDOM_SEED}")
print("✓ Python random seed set")
print("✓ NumPy random seed set")
print("✓ PyTorch random seed set")
print("✓ PyTorch backends set to deterministic mode")
print("="*70 + "\n")

def reset_seeds(seed=RANDOM_SEED):
    """Reset all random seeds to ensure reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

#import data loading and pre-processing functions from separate .py files
from data_loading_v5 import define_metadata, import_exp_data, filter_dataframes, process_dataframes, convert_reader_data, subtract,normalize 
from plot_timecourses import plottimecourselist, plottimecoursearray, figure_layout 
from crosstalk import crosstalk, mixed_crosstalk, antibiotic_crosstalk
from dose_response_fitting import dose_response_fitting
from updated_mechanistic_model import difeq_newest_test_updated, fp_total_timecourse, sensor_fit, plot_fittings
from VAE import VAE, train, test, warmup_scheduler, get_latent_variables, count_parameters
from VAEMLP import MLP, CombinedModel, combo_train, validate

RUN_TAG = "newmech_diff_params_new_run"
tag = f"_{RUN_TAG}" if RUN_TAG else ""

# ============================================================
# IMPROVED FITTING FUNCTIONS (inline instead of separate file)
# ============================================================

def sensor_fit_improved(sensors, samples, tspan, inputs, alpha, K, hill, 
                       algorithm='trf', bounds_version='restrictive'):
    """
    Improved sensor fitting with configurable bounds and algorithms.
    """
    t0 = timer.perf_counter()
    
    inits_train = samples[:, 0::len(tspan)]
    samples_flat = samples.ravel()
    n = sensors
    
    # Reference parameters
    mu_mean = 0.756
    ds_mean = 0.2
    r0_mean = 0.3
    k_mean = 1
    ks_mean = 0.198/2
    theta_mean = 3
    
    # CRITICAL: Different dp0 bounds based on version
    if bounds_version == 'restrictive':
        dp_mean = 0.05
        bounds_lower = [0.1, 0, 0, 0.0001, 0.02, 0.01, 0.6] * n
        bounds_upper = [2, 0.5, 1, 1, 0.2, 1, 6] * n
        print("✓ Using RESTRICTIVE bounds: dp0=[0.02, 0.2]")
        
    elif bounds_version == 'moderate':
        dp_mean = 0.08
        bounds_lower = [0.1, 0, 0, 0.0001, 0.03, 0.01, 0.6] * n
        bounds_upper = [2, 0.8, 1, 1, 0.3, 1, 6] * n
        print("✓ Using MODERATE bounds: dp0=[0.03, 0.3]")
        
    else:  # flexible
        dp_mean = 0.1
        bounds_lower = [0.1, 0, 0, 0.0001, 0, 0.01, 0.6] * n
        bounds_upper = [2, 1, 1, 1, 1, 1, 6] * n
        print("✓ Using FLEXIBLE bounds: dp0=[0, 1]")
    
    para_ref = np.array([mu_mean, ds_mean, r0_mean, k_mean, dp_mean, 
                        ks_mean, theta_mean] * n)
    bounds = np.array([bounds_lower, bounds_upper])
    
    def residual_function(params, y_data, time_range, p_0, inputs, alpha, K, hill):
        params_reshape = params.reshape(n, 7)
        results = []
        
        for i in range(len(p_0)):
            od_0 = p_0[i, 0] / n
            fluor_0 = p_0[i, 1:] / od_0
            yinit = np.zeros(2*n)
            yinit[:n] = od_0
            yinit[n:] = fluor_0
            s = inputs[i]
            
            sol = solve_ivp(
                lambda t, y: difeq_newest_test_updated(t, y, params_reshape, 
                                                       alpha, K, hill, s, n),
                [time_range[0], time_range[-1]], yinit, t_eval=time_range
            )
            
            results.append(fp_total_timecourse(sol.y, n, n).ravel())
        
        results = np.hstack(results)
        residuals = results - y_data
        return residuals
    
    if algorithm == 'differential_evolution':
        print("✓ Using DIFFERENTIAL EVOLUTION (global optimizer)")
        
        def objective(params):
            residuals = residual_function(params, samples_flat, tspan, inits_train, 
                                         inputs, alpha, K, hill)
            return np.sum(residuals**2)
        
        result = differential_evolution(
            objective,
            bounds=list(zip(bounds[0], bounds[1])),
            maxiter=100,
            popsize=15,
            seed=RANDOM_SEED,  # Use global seed
            disp=True,
            workers=1  # Single worker for reproducibility
        )
        sens_popt = result.x
        cost = result.fun
        nfev = result.nfev
        
    else:
        print(f"✓ Using {algorithm.upper()} algorithm")
        result = least_squares(
            fun=residual_function,
            x0=para_ref,
            bounds=bounds,
            method=algorithm,
            args=(samples_flat, tspan, inits_train, inputs, alpha, K, hill),
            verbose=2,
            max_nfev=300
        )
        sens_popt = result.x
        cost = result.cost
        nfev = result.nfev
    
    t1 = timer.perf_counter()
    print(f"✓ Fitting time = {t1 - t0:.0f} s")
    
    return sens_popt, cost, nfev


def verify_fitted_parameters(all_params_test, sensors, mu_mean=0.756):
    """Comprehensive parameter verification."""
    print("\n" + "="*70)
    print("FITTED PARAMETER VERIFICATION:")
    print("="*70)
    
    params_reshaped = all_params_test.reshape(sensors, 7)
    
    fitted_mu = params_reshaped[:, 0]
    fitted_ds = params_reshaped[:, 1]
    fitted_r0 = params_reshaped[:, 2]
    fitted_k = params_reshaped[:, 3]
    fitted_dp0 = params_reshaped[:, 4]
    fitted_Ks = params_reshaped[:, 5]
    fitted_theta = params_reshaped[:, 6]
    
    print(f"\nNumber of sensors: {sensors}")
    print("\n--- Growth Parameters ---")
    print(f"μ (growth rate):     {fitted_mu}")
    print(f"  Mean: {np.mean(fitted_mu):.4f}, Range: [{np.min(fitted_mu):.4f}, {np.max(fitted_mu):.4f}]")
    
    print(f"\nds (input burden):   {fitted_ds}")
    print(f"  Mean: {np.mean(fitted_ds):.4f}, Range: [{np.min(fitted_ds):.4f}, {np.max(fitted_ds):.4f}]")
    
    print("\n--- Protein Decay Parameters (KEY CHECK) ---")
    print(f"dp0 (non-dilution decay): {fitted_dp0}")
    print(f"  Mean: {np.mean(fitted_dp0):.4f}, Range: [{np.min(fitted_dp0):.4f}, {np.max(fitted_dp0):.4f}]")
    
    half_life_stationary = np.log(2) / (fitted_dp0 + 1e-10)
    half_life_exponential = np.log(2) / (np.mean(fitted_mu) + fitted_dp0)
    
    print(f"\nProtein half-lives (stationary phase):")
    print(f"  Mean: {np.mean(half_life_stationary):.2f} hours")
    print(f"\nProtein half-lives (exponential phase):")
    print(f"  Mean: {np.mean(half_life_exponential):.2f} hours")
    
    # Sanity checks
    print("\n--- Sanity Checks ---")
    warnings = []
    
    if np.any(fitted_dp0 < 0.01):
        warnings.append("⚠️  Some dp0 < 0.01 (near lower bound)")
    if np.any(fitted_dp0 > 0.25):
        warnings.append("⚠️  Some dp0 > 0.25 (near upper bound)")
    if np.any(half_life_stationary > 100):
        warnings.append("⚠️  Some proteins have half-life > 100 hours")
    
    if warnings:
        print("\n".join(warnings))
    else:
        print("✅ All parameters look reasonable!")
    
    print("\n" + "="*70)
    
    return params_reshaped


# ============================================================
# MAIN PIPELINE STARTS HERE
# ============================================================

#select which microbial community dataset to work with
community='aTc_IPTG'

#if using antibiotic data, indicate the plasmid and inhibitor combination
plasmid='HSGBla'
inhibitor='TAZ'
if community!='antibiotic_data':
    plasmid=None
    inhibitor=None

#import metadata for selected community
files,readers, fluors, fluor1, fluor2, fluor3, single_file, sensor_names, sensors,time_vector,od_raws,input_arrays,input_names,fp1_raws,fp2_raws,fp3_raws = define_metadata(community,plasmid,inhibitor)

if community=='antibiotic_data':
    od_conv=od_raws
    fp1_conv=fp1_raws
    fp2_conv=fp2_raws
    fp3_conv=None
else:
    od_conv=convert_reader_data(readers, None, od_raws,community)
    fp1_conv=convert_reader_data(readers, fluor1, fp1_raws,community)
    fp2_conv=convert_reader_data(readers, fluor2, fp2_raws,community)
    if sensors==3:
        fp3_conv=convert_reader_data(readers, fluor3, fp3_raws,community)
    else:
        fp3_conv=None

# Subtract basal expression
subtracted_fp1_conv_all, subtracted_fp2_conv_all, subtracted_fp3_conv_all=subtract(community,time_vector,fp1_conv,fp2_conv,fp3_conv,sensors,input_arrays)

# Min-max scale
normalized_fp1_conv_all, normalized_fp2_conv_all, normalized_fp3_conv_all=normalize(subtracted_fp1_conv_all,subtracted_fp2_conv_all,subtracted_fp3_conv_all)

#Append the full OD and fluorescence timecourses
od_stack=np.empty((0,len(time_vector)), dtype=float)
for i, reader in enumerate(fluor1):
    od_stack=np.vstack((od_stack,od_conv[i]))

concat_list = [od_stack]
concat_list.append(normalized_fp1_conv_all)
concat_list.append(normalized_fp2_conv_all)
if normalized_fp3_conv_all is not None and normalized_fp3_conv_all.size>0:
    concat_list.append(normalized_fp3_conv_all)
exp_data_new=np.concatenate(concat_list,axis=1)
    
exp_inputs=np.vstack(list(input_arrays.values()))

# Load pre-calculated parameters
timepoint=20 
if community=='antibiotic_data':
    save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
else:
    save_dir = f"parameter_files/{community}/"

alpha_files=glob.glob(f'{save_dir}*_{timepoint}hr_{community}_alphas_mixed.npy')
alphas=np.load(max(alpha_files))

hill_timepoint=20 
K_files=glob.glob(f'{save_dir}*_{hill_timepoint}hr_{community}_K_calc.npy')
K_calc=np.load(max(K_files))
    
hill_files=glob.glob(f'{save_dir}*_{hill_timepoint}hr_{community}_hill_coef.npy')
hill_calc=np.load(max(hill_files))

# Calculate limits
true_K=np.zeros(sensors)
upper=np.zeros(sensors)
lower=np.zeros(sensors)
for i in range(sensors):
    true_K[i]=K_calc[i*sensors + i] 
    upper[i]=true_K[i]*(99**(1/hill_calc[i*sensors + i]))
    lower[i]=true_K[i]/(99**(1/hill_calc[i*sensors + i])) 

# Filter data
if ((community=='cuma_ohc_atc')|(community=='van_dapg_nar')|(community=='antibiotic_data')):
    mask=(exp_inputs>0).all(axis=1)
else:
    mask = (exp_inputs >= lower).all(axis=1) & (exp_inputs <= upper).all(axis=1) 

filtered_exp_inputs = exp_inputs[mask]
filtered_exp_data_new = exp_data_new[mask]

# Normalize
normalized_exp_inputs = filtered_exp_inputs / true_K
normalized_K_calc=K_calc.reshape((sensors,sensors))/true_K


# ============================================================
# PARAMETER FITTING WITH MULTIPLE ALGORITHMS
# ============================================================

print("\n" + "="*70)
print("ALTERNATIVE-FIT ANALYSIS")
print("Generating multiple parameter sets using different algorithms")
print(f"Results will be saved with tag: {tag}")
print("="*70)

alpha_reshaped = alphas.reshape((sensors, sensors))
hill_reshaped = hill_calc.reshape((sensors, sensors))

# Store all parameter sets
all_parameter_sets = {}
all_costs = {}

# Configuration
fit_configurations = [
    ('trf', 'restrictive', 'TRF_Restrictive'),
    ('dogbox', 'moderate', 'Dogbox_Moderate'),
    ('trf', 'moderate', 'TRF_Moderate'),
]

# Run fitting
for algorithm, bounds_version, name in fit_configurations:
    print(f"\n{'='*70}")
    print(f"Fitting with: {name}")
    print(f"Algorithm: {algorithm}, Bounds: {bounds_version}")
    print(f"{'='*70}\n")
    
    params, cost, nfev = sensor_fit_improved(
        sensors, 
        filtered_exp_data_new, 
        time_vector,
        normalized_exp_inputs, 
        alpha_reshaped, 
        normalized_K_calc, 
        hill_reshaped,
        algorithm=algorithm,
        bounds_version=bounds_version
    )
    
    all_parameter_sets[name] = params
    all_costs[name] = cost
    
    print(f"\n>>> Verifying {name} parameters:")
    params_reshaped = verify_fitted_parameters(params, sensors)
    
    # Save with RUN_TAG
    if community == 'antibiotic_data':
        save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
    else:
        save_dir = f"parameter_files/{community}/"
    
    os.makedirs(save_dir, exist_ok=True)
    datestamp = datetime.now().strftime("%Y-%m-%d")
    
    np.save(f"{save_dir}{datestamp}{tag}_params_{name}.npy", params)
    np.save(f"{save_dir}{datestamp}{tag}_cost_{name}.npy", cost)
    print(f"✓ Saved: {save_dir}{datestamp}{tag}_params_{name}.npy")

# Print comparison
print("\n" + "="*70)
print("FIT QUALITY COMPARISON:")
print("="*70)
for name, cost in all_costs.items():
    print(f"  {name:30s}: Cost = {cost:.6f}")
print("="*70 + "\n")

print(f"✅ Successfully generated {len(all_parameter_sets)} parameter sets!")
print(f"   All saved with tag: {tag}")


# ============================================================
# GENERATE SIMULATIONS FOR EACH PARAMETER SET
# ============================================================

from scipy.stats import truncnorm

print("\n" + "="*70)
print("GENERATING 10K SIMULATIONS FOR EACH PARAMETER SET")
print("="*70)

# Calculate initial conditions
inits_train = filtered_exp_data_new[:, 0::len(time_vector)]

od_0_mean = np.mean(inits_train[:, 0]) / sensors
od_0_std = np.std(inits_train[:, 0]) / sensors
od_0_max = np.max(inits_train[:, 0]) / sensors
od_0_min = np.min(inits_train[:, 0]) / sensors

fluor_stats = []
for j in range(sensors):
    od_0 = inits_train[:, 0] / sensors
    fluor_mean = np.mean(inits_train[:, j+1] / od_0)
    fluor_std = np.std(inits_train[:, j+1] / od_0)
    fluor_min = np.min(inits_train[:, j+1] / od_0)
    fluor_max = np.max(inits_train[:, j+1] / od_0)
    fluor_stats.append((fluor_mean, fluor_std, fluor_min, fluor_max))

def get_truncated_normal(mean=0, sd=1, low=0, upp=10):
    return truncnorm((low - mean) / sd, (upp - mean) / sd, loc=mean, scale=sd)

num_simulations = 10000

# Generate same inputs for all parameter sets (with explicit seed)
np.random.seed(RANDOM_SEED)
s_full = (10 ** ((np.random.rand(num_simulations, sensors) * 
          (np.log10(upper) - np.log10(lower))) + np.log10(lower))) / true_K

print(f"Generated {num_simulations} input combinations with seed={RANDOM_SEED}")

# Generate simulations
all_simulations = {}
all_y0_values = {}

for param_name, all_params_test in all_parameter_sets.items():
    print(f"\n>>> Generating simulations for: {param_name}")
    
    # Reset seed for consistent initial conditions across parameter sets
    np.random.seed(RANDOM_SEED)
    
    optimized_params = all_params_test.reshape(sensors, 7)
    time_courses = np.empty((num_simulations, sensors+1, len(time_vector)))
    y0_values = []
    
    for i in range(num_simulations):
        if (i+1) % 2000 == 0:
            print(f"    Progress: {i+1}/{num_simulations}")
        
        s = s_full[i]
        
        OD0 = get_truncated_normal(mean=od_0_mean, sd=od_0_std, 
                                   low=od_0_min, upp=od_0_max).rvs()
        
        fluor_values = [
            get_truncated_normal(mean=stats[0], sd=stats[1], 
                               low=stats[2], upp=stats[3]).rvs()
            for stats in fluor_stats
        ]
        
        y0 = np.hstack(([OD0] * sensors, fluor_values))
        
        sol = solve_ivp(
            difeq_newest_test_updated,
            [time_vector[0], time_vector[-1]],
            y0,
            t_eval=time_vector,
            args=(optimized_params, alpha_reshaped, normalized_K_calc, 
                  hill_reshaped, s, sensors)
        )
        
        time_courses[i] = fp_total_timecourse(sol.y, sensors, sensors)
        y0_values.append(y0)
    
    all_simulations[param_name] = time_courses
    all_y0_values[param_name] = np.array(y0_values)
    
    # Save with RUN_TAG
    if community == 'antibiotic_data':
        save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
    else:
        save_dir = f"parameter_files/{community}/"
    
    os.makedirs(save_dir, exist_ok=True)
    
    np.savez(f"{save_dir}{datestamp}{tag}_{community}_curves10k_{param_name}.npz", 
             time_courses)
    np.save(f"{save_dir}{datestamp}{tag}_{community}_S_10k_{param_name}.npy", s_full)
    np.save(f"{save_dir}{datestamp}{tag}_{community}_y0_10k_{param_name}.npy", 
            np.array(y0_values))
    
    print(f"✓ Saved: {save_dir}{datestamp}{tag}_{community}_curves10k_{param_name}.npz")

print("\n" + "="*70)
print("✅ ALL SIMULATIONS GENERATED SUCCESSFULLY")
print(f"   Total simulations: {len(all_parameter_sets)} x 10,000 = {len(all_parameter_sets) * 10000}")
print(f"   All files saved with tag: {tag}")
print("="*70 + "\n")

print("\nREADY FOR VAE-MLP TRAINING!")
print(f"Next: Train VAE-MLP on each parameter set's simulations")
print(f"Files to use:")
for param_name in all_parameter_sets.keys():
    print(f"  - {datestamp}{tag}_{community}_curves10k_{param_name}.npz")


# ============================================================
# VAE-MLP TRAINING FOR EACH PARAMETER SET
# ============================================================

print("\n" + "="*70)
print("TRAINING SEPARATE VAE-MLP FOR EACH PARAMETER SET")
print("This demonstrates parameter non-uniqueness doesn't affect predictions")
print("="*70)

import time as timer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Hyperparameters
batch_size = 32
latent_dim = 10
latent_channel = 6
alpha_vae = 1e-4
lr = 1e-3
min_lr = 5e-6
epochs = 400
gamma = 0.99
weight_decay = 1e-5
warmup_epochs = 8
patience = 30
hidden_size = 128

# Store all trained models
trained_models = {}
training_histories = {}

# Train VAE-MLP for each parameter set
for param_name in all_parameter_sets.keys():
    print(f"\n{'='*70}")
    print(f"TRAINING VAE-MLP FOR: {param_name}")
    print(f"{'='*70}\n")
    
    t0 = timer.perf_counter()
    
    # Load simulations for this parameter set
    if community == 'antibiotic_data':
        save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
    else:
        save_dir = f"parameter_files/{community}/"
    
    data_array = np.load(f"{save_dir}{datestamp}{tag}_{community}_curves10k_{param_name}.npz")['arr_0']
    s_full_loaded = np.load(f"{save_dir}{datestamp}{tag}_{community}_S_10k_{param_name}.npy")
    
    # Flatten data
    data_concat = data_array.reshape(data_array.shape[0], -1)
    seq_length = data_concat.shape[1]
    
    # Combine with experimental data
    combined_data_np = np.vstack((data_concat, filtered_exp_data_new))
    
    # Create indices to locate experimental data
    exp_data_indices = range(data_concat.shape[0], 
                             data_concat.shape[0] + filtered_exp_data_new.shape[0])
    
    # Normalize
    scaler = MinMaxScaler()
    data_normalized_np = scaler.fit_transform(combined_data_np)
    data_normalized = torch.tensor(data_normalized_np).float().unsqueeze(1)
    
    # Prepare inputs
    combined_inputs = np.vstack((s_full_loaded, normalized_exp_inputs))
    log_inputs = np.log10(combined_inputs)
    log_inputs_tensor = torch.from_numpy(log_inputs).float()
    
    # Split train/test (SAME split for all models for fair comparison)
    # Explicitly set random_state for reproducibility
    train_data, test_data, train_labels, test_labels, train_idx, test_idx = train_test_split(
        data_normalized, log_inputs_tensor, 
        range(data_normalized.shape[0]), 
        test_size=0.2, random_state=RANDOM_SEED
    )
    
    print(f"  Train/test split: {len(train_idx)}/{len(test_idx)} samples")
    
    # ============================================================
    # TRAIN VAE
    # ============================================================
    
    print(f">>> Training VAE...")
    
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    
    # Reset all seeds before model initialization for reproducibility
    reset_seeds(RANDOM_SEED)
    
    vae_model = VAE(latent_dim=latent_dim, latent_channel=latent_channel, seq_length=seq_length)
    vae_model = vae_model.to(device)
    
    print(f'VAE has {count_parameters(vae_model):,} parameters')
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(vae_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    train_loss_values = []
    test_loss_values = []
    
    best_test_loss = np.inf
    epochs_no_improve = 0
    
    scheduler1 = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda epoch: warmup_scheduler(epoch, warmup_epochs))
    scheduler2 = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    for epoch in range(epochs):
        train_loss = train(vae_model, train_loader, optimizer, criterion, 
                          alpha_vae, device, latent_channel, seq_length)
        test_loss = test(vae_model, test_loader, criterion, device, 
                        latent_channel, seq_length)
        train_loss_values.append(train_loss)
        test_loss_values.append(test_loss)
        
        # Clamp minimum learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = max(param_group['lr'], min_lr)
        
        interval = 2 if epoch < 10 else 40
        if (epoch + 1) % interval == 0:
            print(f'Epoch: {epoch + 1} Train: {train_loss:.7f}, Test: {test_loss:.7f}, Lr:{param_group["lr"]:.8f}')
        
        # Update learning rate
        if epoch < warmup_epochs:
            scheduler1.step()
        else:
            scheduler2.step()
        
        # Early stopping
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve == patience:
            print('Early stopping!')
            break
    
    # Save VAE model
    if community == 'antibiotic_data':
        model_save_dir = f"final_trained_models/{community}/{plasmid}_{inhibitor}/"
    else:
        model_save_dir = f"final_trained_models/{community}/"
    
    os.makedirs(model_save_dir, exist_ok=True)
    vae_path = f"{model_save_dir}{datestamp}{tag}_VAE_{community}_{param_name}.pt"
    torch.save(vae_model.state_dict(), vae_path)
    print(f"✓ Saved VAE: {vae_path}")
    
    # ============================================================
    # TRAIN MLP
    # ============================================================
    
    print(f"\n>>> Training MLP...")
    
    # Reset seeds before MLP initialization
    reset_seeds(RANDOM_SEED)
    
    # Create MLP
    mlp_model = MLP(latent_dim, hidden_size, log_inputs_tensor.shape[1]).to(device)
    
    # Load the trained VAE
    vae_model_for_combo = VAE(latent_dim, latent_channel, seq_length).to(device)
    vae_model_for_combo.load_state_dict(torch.load(vae_path, weights_only=True))
    
    # Create combined model
    combined_model = CombinedModel(vae_model_for_combo, mlp_model).to(device)
    
    print(f'MLP has {count_parameters(mlp_model):,} parameters')
    print(f'Combined model has {count_parameters(combined_model):,} parameters')
    
    # Prepare data loaders
    train_dataset = TensorDataset(train_data, train_labels)
    train_loader_mlp = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)  # No shuffle for reproducibility
    test_dataset = TensorDataset(test_data, test_labels)
    test_loader_mlp = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(mlp_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_test_loss = np.inf
    epochs_no_improve = 0
    
    scheduler1 = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda epoch: warmup_scheduler(epoch, warmup_epochs))
    scheduler2 = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    train_loss_values_mlp = []
    test_loss_values_mlp = []
    
    for epoch in range(epochs):
        train_loss = combo_train(combined_model, train_loader_mlp, optimizer, 
                                criterion, device, latent_channel, seq_length)
        test_loss = validate(combined_model, test_loader_mlp, criterion, 
                           device, latent_channel, seq_length)
        train_loss_values_mlp.append(train_loss)
        test_loss_values_mlp.append(test_loss)
        
        # Clamp minimum learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = max(param_group['lr'], min_lr)
        
        interval = 2 if epoch < 10 else 40
        if (epoch + 1) % interval == 0:
            print(f'Epoch: {epoch + 1} Train: {train_loss:.7f}, Test: {test_loss:.7f}, Lr:{param_group["lr"]:.8f}')
        
        # Update learning rate
        if epoch < warmup_epochs:
            scheduler1.step()
        else:
            scheduler2.step()
        
        # Early stopping
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve == patience:
            print('Early stopping!')
            break
    
    # Save combined model
    combined_path = f"{model_save_dir}{datestamp}{tag}_VAEMLP_{community}_{param_name}.pt"
    torch.save(combined_model.state_dict(), combined_path)
    print(f"✓ Saved VAE-MLP: {combined_path}")
    
    # Store for later comparison
    trained_models[param_name] = {
        'model': combined_model,
        'scaler': scaler,
        'train_idx': train_idx,
        'test_idx': test_idx,
        'exp_data_indices': exp_data_indices,
        'vae_train_loss': train_loss_values,
        'vae_test_loss': test_loss_values,
        'mlp_train_loss': train_loss_values_mlp,
        'mlp_test_loss': test_loss_values_mlp,
        'data_normalized': data_normalized,
        'log_inputs_tensor': log_inputs_tensor
    }
    
    training_histories[param_name] = {
        'vae_train': train_loss_values,
        'vae_test': test_loss_values,
        'mlp_train': train_loss_values_mlp,
        'mlp_test': test_loss_values_mlp
    }
    
    t1 = timer.perf_counter()
    print(f"\n✓ Total training time for {param_name}: {(t1-t0)/60:.1f} minutes")

print("\n" + "="*70)
print("✅ ALL VAE-MLPs TRAINED SUCCESSFULLY")
print(f"   Trained {len(trained_models)} models")
print("="*70 + "\n")


# ============================================================
# CALCULATE R² ON EXPERIMENTAL DATA ONLY
# ============================================================

print("\n" + "="*70)
print("CALCULATING R² ON EXPERIMENTAL DATA ONLY")
print("(Stricter test - only real experimental samples)")
print("="*70)

# Custom function to round to the nearest 0.5
def round_to_nearest_half(value):
    return round(value * 2) / 2

all_r2_exp_train = {}
all_r2_exp_test = {}

for param_name, model_info in trained_models.items():
    print(f"\n>>> Calculating experimental R² for {param_name}...")
    
    model = model_info['model']
    scaler = model_info['scaler']
    train_idx = model_info['train_idx']
    test_idx = model_info['test_idx']
    exp_data_indices = model_info['exp_data_indices']
    data_normalized = model_info['data_normalized']
    log_inputs_tensor = model_info['log_inputs_tensor']
    
    # Create masks for experimental data only
    train_mask = np.isin(train_idx, list(exp_data_indices))
    test_mask = np.isin(test_idx, list(exp_data_indices))
    
    # Get experimental data from train/test splits
    mlp_train_exp_curves = data_normalized[train_idx][train_mask]
    mlp_test_exp_curves = data_normalized[test_idx][test_mask]
    train_exp_inputs = log_inputs_tensor[train_idx][train_mask]
    test_exp_inputs = log_inputs_tensor[test_idx][test_mask]
    
    # Get predictions on experimental data only
    with torch.no_grad():
        output_exp_train = model(mlp_train_exp_curves.to(device), latent_channel, seq_length)
        output_exp_test = model(mlp_test_exp_curves.to(device), latent_channel, seq_length)
    
    # Squeeze and convert
    output_exp_train = output_exp_train.squeeze(1)
    output_exp_test = output_exp_test.squeeze(1)
    
    # Convert from log space
    subset_train_data = np.log10(true_K * 10**train_exp_inputs.cpu().numpy())
    subset_test_data = np.log10(true_K * 10**test_exp_inputs.cpu().numpy())
    output_train_exp = np.log10(true_K * 10**output_exp_train.cpu().numpy())
    output_test_exp = np.log10(true_K * 10**output_exp_test.cpu().numpy())
    
    # Calculate R² for each input
    r2_exp_train_per_input = []
    r2_exp_test_per_input = []
    
    for col in range(sensors):
        r2_train = r2_score(subset_train_data[:, col], output_train_exp[:, col])
        r2_test = r2_score(subset_test_data[:, col], output_test_exp[:, col])
        r2_exp_train_per_input.append(r2_train)
        r2_exp_test_per_input.append(r2_test)
        
        print(f"  {input_names[col]}: Exp Train R² = {r2_train:.4f}, Exp Test R² = {r2_test:.4f}")
    
    all_r2_exp_train[param_name] = r2_exp_train_per_input
    all_r2_exp_test[param_name] = r2_exp_test_per_input
    
    # Store for plotting
    model_info['exp_train_predictions'] = output_train_exp
    model_info['exp_test_predictions'] = output_test_exp
    model_info['exp_train_true'] = subset_train_data
    model_info['exp_test_true'] = subset_test_data


# ============================================================
# SEPARATE PANEL GENERATION
# ============================================================

print("\n" + "="*70)
print("GENERATING SEPARATE PANELS")
print("="*70)

# Set up save directory
if community == 'antibiotic_data':
    fig_save_dir = f"figures/{community}/{plasmid}_{inhibitor}/"
else:
    fig_save_dir = f"figures/{community}/"

os.makedirs(fig_save_dir, exist_ok=True)

# Define model names and abbreviated names
model_names = list(trained_models.keys())
abbreviated_names = ['TRF_R', 'Dogbox', 'TRF_M']  # Abbreviated versions

# Helper function for tick generation
def three_ticks(min_val, max_val):
    mid = 0.5 * (min_val + max_val)
    ticks = [
        round_to_nearest_half(min_val),
        round_to_nearest_half(mid),
        round_to_nearest_half(max_val),
    ]
    ticks = sorted(set(ticks))
    if len(ticks) == 3:
        return ticks
    center = round_to_nearest_half(mid)
    return [center - 0.5, center, center + 0.5]

# ============================================================
# PANEL A: SCATTER PLOTS (REDUCED VERTICAL SPACING)
# ============================================================

print("\n>>> Creating Panel A (scatter plots)...")

# Font scale = 2x
font_scale = 2.0

# Calculate figure size - LARGER subplots
n_cols = len(model_names)
n_rows = sensors

# INCREASED base size per subplot
subplot_width = 6.0
subplot_height = 6.0

fig_width = subplot_width * n_cols
fig_height = subplot_height * n_rows

# Create figure with MINIMAL vertical spacing
fig = plt.figure(figsize=(fig_width, fig_height))
gs = GridSpec(n_rows, n_cols, hspace=0.15, wspace=0.3)

# Scaled sizes
scatter_size = 40 * font_scale
line_width = 3.5 * font_scale
spine_width = 2.5 * font_scale

r2_fontsize = int(13 * font_scale)
title_fontsize = int(14 * font_scale)
label_fontsize = int(13 * font_scale)
tick_fontsize = int(11 * font_scale)

for c, param_name in enumerate(model_names):
    if param_name not in trained_models:
        continue
    
    model_info = trained_models[param_name]
    test_predictions = model_info['exp_test_predictions']
    test_true = model_info['exp_test_true']
    
    for r in range(sensors):
        ax = fig.add_subplot(gs[r, c])

        true_vals = test_true[:, r]
        pred_vals = test_predictions[:, r]

        ax.scatter(true_vals, pred_vals, s=scatter_size, color="blue", alpha=0.6)

        data_min = min(true_vals.min(), pred_vals.min())
        data_max = max(true_vals.max(), pred_vals.max())

        if input_names[r].lower() == "atc":
            base = -1.8
            ticks = [base, base + 0.5, base + 1.0]
            mn = min(data_min, base)
            mx = max(data_max, base + 1.0)
        else:
            mn = data_min
            mx = data_max
            ticks = three_ticks(mn, mx)

        pad = 0.05 * (mx - mn)

        ax.plot([mn, mx], [mn, mx], color="red", lw=line_width, alpha=0.5)

        ax.set_xticks(ticks)
        ax.set_yticks(ticks)
        ax.set_xlim(mn - pad, mx + pad)
        ax.set_ylim(mn - pad, mx + pad)
        ax.set_aspect("equal", adjustable="box")

        r2 = all_r2_exp_test[param_name][r]
        ax.text(
            0.05, 0.95, f"R² = {r2:.3f}",
            transform=ax.transAxes, va="top", ha="left",
            fontsize=r2_fontsize
        )

        # Only add title to top row - use ABBREVIATED names
        if r == 0:
            ax.set_title(abbreviated_names[c], fontsize=title_fontsize, pad=10)
        
        # Only add y-label to leftmost column
        if c == 0:
            ax.set_ylabel(f"Pred log({input_names[r]})", fontsize=label_fontsize)
        
        # Only add x-label to bottom row
        if r == sensors - 1:
            ax.set_xlabel(f"True log({input_names[r]})", fontsize=label_fontsize)

        ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)
        
        for spine in ax.spines.values():
            spine.set_linewidth(spine_width)

plt.tight_layout()

# Save Panel A
for ext in ['svg', 'png', 'pdf']:
    filepath = f"{fig_save_dir}panel_A_scatter{tag}.{ext}"
    if ext == 'png':
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
    else:
        fig.savefig(filepath, bbox_inches='tight')
    print(f"  ✓ Saved Panel A ({ext}): {filepath}")

plt.close(fig)

# ============================================================
# PANEL B: CORRELATION MATRICES (ABBREVIATED LABELS)
# ============================================================

print("\n>>> Creating Panel B (correlation matrices)...")

# Calculate figure size - make it match Panel A width
subplot_width = 6.0  # Match Panel A subplot width
subplot_height = 6.0  # Match Panel A subplot height

fig_width = subplot_width
fig_height = subplot_height * sensors

# Create figure with MINIMAL vertical spacing
fig = plt.figure(figsize=(fig_width, fig_height))
gs = GridSpec(sensors, 1, hspace=0.25)

# Scaled sizes for Panel B
corr_fontsize = int(12 * font_scale)
corr_label_fontsize = int(11 * font_scale)
corr_title_fontsize = int(12 * font_scale)
grid_line_width = 2 * font_scale

for r in range(sensors):
    ax = fig.add_subplot(gs[r, 0])

    # Calculate correlation matrix
    corr = np.ones((len(model_names), len(model_names)))
    for i, n1 in enumerate(model_names):
        if n1 not in trained_models:
            continue
        for j, n2 in enumerate(model_names):
            if n2 not in trained_models:
                continue
            p1 = trained_models[n1]['exp_test_predictions'][:, r]
            p2 = trained_models[n2]['exp_test_predictions'][:, r]
            corr[i, j] = np.corrcoef(p1, p2)[0, 1]

    # Invisible background
    ax.imshow(np.ones_like(corr), cmap="Greys", alpha=0)

    # Grid lines
    for i in range(len(model_names) + 1):
        ax.axhline(i - 0.5, color="black", linewidth=grid_line_width)
        ax.axvline(i - 0.5, color="black", linewidth=grid_line_width)

    # Text values in cells
    for i in range(len(model_names)):
        for j in range(len(model_names)):
            ax.text(j, i, f"{corr[i, j]:.3f}",
                    ha="center", va="center", 
                    fontsize=corr_fontsize)

    # Use ABBREVIATED names and NO rotation
    ax.set_xticks(range(len(model_names)))
    ax.set_yticks(range(len(model_names)))
    ax.set_xticklabels(abbreviated_names, rotation=0, ha="center",
                      fontsize=corr_label_fontsize)
    ax.set_yticklabels(abbreviated_names, fontsize=corr_label_fontsize)
    
    ax.set_title(
        f"{input_names[r]}",
        fontsize=corr_title_fontsize, pad=16
    )

plt.tight_layout()

# Save Panel B
for ext in ['svg', 'png', 'pdf']:
    filepath = f"{fig_save_dir}panel_B_correlation{tag}.{ext}"
    if ext == 'png':
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
    else:
        fig.savefig(filepath, bbox_inches='tight')
    print(f"  ✓ Saved Panel B ({ext}): {filepath}")

plt.close(fig)


# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\n" + "="*70)
print("COMPLETE SUMMARY: R² Statistics")
print("="*70)

for sensor_idx in range(sensors):
    print(f"\n{input_names[sensor_idx]}:")
    
    print(f"\n  Experimental Data Only (Used for Panels A & B):")
    print(f"    Training R²: {np.mean([all_r2_exp_train[name][sensor_idx] for name in trained_models.keys()]):.4f} ± {np.std([all_r2_exp_train[name][sensor_idx] for name in trained_models.keys()]):.4f}")
    print(f"    Test R²:     {np.mean([all_r2_exp_test[name][sensor_idx] for name in trained_models.keys()]):.4f} ± {np.std([all_r2_exp_test[name][sensor_idx] for name in trained_models.keys()]):.4f}")

print(f"\n" + "="*70)
print("CONCLUSION:")
print("="*70)

avg_exp_test_std = np.mean([np.std([all_r2_exp_test[name][i] for name in trained_models.keys()]) 
                            for i in range(sensors)])

if avg_exp_test_std < 0.05:
    print("✅ Low variability in R² scores across parameter sets!")
    print("✅ Different parameter sets yield similar prediction performance!")
    print("✅ Parameter non-uniqueness does NOT affect input predictions!")
    print("\nThis demonstrates that despite having different ODE parameters,")
    print("the VAE-MLP models all make similar predictions on experimental test data.")
    print("Therefore, parameter non-uniqueness is NOT a problem for this application!")
else:
    print("⚠️  Significant variability in predictions across parameter sets.")
    print(f"   Exp Test std: {avg_exp_test_std:.4f}")
    print("   Consider investigating parameter sensitivity further.")

print("="*70 + "\n")

print("\n" + "="*70)
print("SEPARATE PANELS GENERATED!")
print("="*70)
print(f"\nFiles saved in: {fig_save_dir}")
print("  Panel A (scatter plots):")
print(f"    • panel_A_scatter{tag}.[svg/png/pdf]")
print("  Panel B (correlation matrices):")
print(f"    • panel_B_correlation{tag}.[svg/png/pdf]")
print("\nFeatures:")
print("  ✓ Using experimental TEST data only (stricter evaluation)")
print("  ✓ Reduced vertical spacing (hspace=0.15 for A, 0.25 for B)")
print("  ✓ Abbreviated labels in Panel B (TRF_R, Dogbox, TRF_M)")
print("  ✓ No rotation for Panel B labels")
print("  ✓ 2x larger fonts throughout")
print("  ✓ Both panels similar width")
print("\n" + "="*70)

print("\n🎉 ALTERNATIVE-FIT ANALYSIS COMPLETE!")
print(f"   Generated {len(all_parameter_sets)} parameter sets")
print(f"   Trained {len(trained_models)} VAE-MLP models")
print(f"   All results saved with tag: {tag}")
print(f"   R² calculated on experimental test data partition")
print("\n" + "="*70)
print("REPRODUCIBILITY MEASURES:")
print("="*70)
print(f"✓ Global random seed: {RANDOM_SEED}")
print("✓ All random number generators seeded:")
print("  • Python random")
print("  • NumPy random")
print("  • PyTorch (CPU and CUDA)")
print("✓ PyTorch deterministic mode enabled")
print("✓ Train/test split fixed (random_state={})".format(RANDOM_SEED))
print("✓ Data loaders without shuffle")
print("✓ Model weights initialized with fixed seed")
print("✓ Simulation initial conditions seeded")
print("\nResults should be identical across runs on the same hardware.")
print("Note: Minor differences may occur across different hardware/CUDA versions.")
print("="*70 + "\n")

REPRODUCIBILITY SETTINGS
Random seed set to: 42
✓ Python random seed set
✓ NumPy random seed set
✓ PyTorch random seed set
✓ PyTorch backends set to deterministic mode


ALTERNATIVE-FIT ANALYSIS
Generating multiple parameter sets using different algorithms
Results will be saved with tag: _newmech_diff_params_new_run

Fitting with: TRF_Restrictive
Algorithm: trf, Bounds: restrictive

✓ Using RESTRICTIVE bounds: dp0=[0.02, 0.2]
✓ Using TRF algorithm
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         1.3725e+03                                    7.57e+03    
       1              2         7.1606e+02      6.56e+02       6.38e-01       4.42e+03    
       2              3         4.7137e+02      2.45e+02       4.03e-01       1.80e+03    
       3              4         4.2152e+02      4.98e+01       4.90e-01       4.18e+03    
       4              5         3.5917e+02      6.23e+01       1.15e-01       9.28e+01    
  

/tmp/ipykernel_95348/4082891139.py:993: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✓ Saved Panel A (svg): figures/aTc_IPTG/panel_A_scatter_newmech_diff_params_new_run.svg
  ✓ Saved Panel A (png): figures/aTc_IPTG/panel_A_scatter_newmech_diff_params_new_run.png
  ✓ Saved Panel A (pdf): figures/aTc_IPTG/panel_A_scatter_newmech_diff_params_new_run.pdf

>>> Creating Panel B (correlation matrices)...
  ✓ Saved Panel B (svg): figures/aTc_IPTG/panel_B_correlation_newmech_diff_params_new_run.svg


/tmp/ipykernel_95348/4082891139.py:1071: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  ✓ Saved Panel B (png): figures/aTc_IPTG/panel_B_correlation_newmech_diff_params_new_run.png
  ✓ Saved Panel B (pdf): figures/aTc_IPTG/panel_B_correlation_newmech_diff_params_new_run.pdf

COMPLETE SUMMARY: R² Statistics

aTc:

  Experimental Data Only (Used for Panels A & B):
    Training R²: 0.9744 ± 0.0042
    Test R²:     0.9251 ± 0.0087

IPTG:

  Experimental Data Only (Used for Panels A & B):
    Training R²: 0.9733 ± 0.0103
    Test R²:     0.9494 ± 0.0165

CONCLUSION:
✅ Low variability in R² scores across parameter sets!
✅ Different parameter sets yield similar prediction performance!
✅ Parameter non-uniqueness does NOT affect input predictions!

This demonstrates that despite having different ODE parameters,
the VAE-MLP models all make similar predictions on experimental test data.
Therefore, parameter non-uniqueness is NOT a problem for this application!


SEPARATE PANELS GENERATED!

Files saved in: figures/aTc_IPTG/
  Panel A (scatter plots):
    • panel_A_scatter_newmech_di